In [1]:
!pip install wandb -q

In [2]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms, models
from pathlib import Path
import shutil
from PIL import Image
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split
import wandb

In [3]:
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

In [4]:
TRAIN_DIR = Path("/kaggle/input/datasets/simonbouchardk/quebec-street-images/quebec-street-images/dataset/train")
TEST_DIR  = Path("/kaggle/input/datasets/simonbouchardk/quebec-street-images/quebec-street-images/dataset/test")

REGIONS   = sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()])
label_map = {r: i for i, r in enumerate(REGIONS)}

print("Copying dataset to local SSD...")
shutil.copytree(TRAIN_DIR, Path("/kaggle/working/train"))
shutil.copytree(TEST_DIR,  Path("/kaggle/working/test"))

TRAIN_DIR = Path("/kaggle/working/train")
TEST_DIR  = Path("/kaggle/working/test")
print("Done")

print(f"{len(REGIONS)} regions: {REGIONS}")

Copying dataset to local SSD...
Done
17 regions: ['Abitibi-Temiscamingue', 'Bas-Saint-Laurent', 'Capitale-Nationale', 'Centre-du-Quebec', 'Chaudiere-Appalaches', 'Cote-Nord', 'Estrie', 'Gaspesie-Iles-de-la-Madeleine', 'Lanaudiere', 'Laurentides', 'Laval', 'Mauricie', 'Monteregie', 'Montreal', 'Nord-du-Quebec', 'Outaouais', 'Saguenay-Lac-Saint-Jean']


In [5]:
config = {
    "model":            "efficientnet_v2_m",
    "input_size":       480,
    "batch_size":          16,
    "accumulation_steps":  2,
    "lr_head":          1e-3,
    "lr_finetune":      5e-5,
    "epochs_head":      5,
    "epochs_finetune":  20,
    "weight_decay":     0.01,
    "label_smoothing":  0.1,
    "dropout":          0.4,
    "num_classes":      len(REGIONS),
}

In [6]:
train_samples = [{"image_id": p.stem, "region": p.parent.name} for p in TRAIN_DIR.rglob("*.jpg")]
test_samples  = [{"image_id": p.stem, "region": p.parent.name} for p in TEST_DIR.rglob("*.jpg")]

train_val_df = pd.DataFrame(train_samples)
test_df      = pd.DataFrame(test_samples)

train_df, val_df = train_test_split(
    train_val_df, test_size=0.111, stratify=train_val_df["region"], random_state=42
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 9642 | Val: 1205 | Test: 1909


In [7]:
CACHE_DIR = Path("/kaggle/working/tensor_cache")
CACHE_DIR.mkdir(exist_ok=True)

def build_disk_cache(df, base_dir, split_name):
    cache_dir = CACHE_DIR / split_name
    cache_dir.mkdir(exist_ok=True)

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Caching {split_name}"):
        out = cache_dir / f"{row.image_id}.pt"
        if out.exists():
            continue
        img = Image.open(base_dir / row.region / f"{row.image_id}.jpg").convert("RGB")
        img = img.resize((512, 512), Image.BILINEAR)
        torch.save(transforms.PILToTensor()(img), out)

build_disk_cache(train_df, TRAIN_DIR, "train")
build_disk_cache(val_df,   TRAIN_DIR, "val")
build_disk_cache(test_df,  TEST_DIR,  "test")

Caching test: 100%|██████████| 1909/1909 [00:29<00:00, 63.70it/s]


In [8]:
normalize   = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
augment     = transforms.Compose([
    transforms.RandomCrop(480),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
])
center_crop = transforms.CenterCrop(480)

class TensorCacheDataset(Dataset):
    def __init__(self, df, split_name, train=False):
        self.df        = df.reset_index(drop=True)
        self.cache_dir = CACHE_DIR / split_name
        self.train     = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = torch.load(self.cache_dir / f"{row.image_id}.pt").float() / 255.0
        img = augment(img) if self.train else center_crop(img)
        img = normalize(img)
        return img, label_map[row.region]

train_loader = DataLoader(TensorCacheDataset(train_df, "train", train=True),  batch_size=config["batch_size"], shuffle=True,  num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(TensorCacheDataset(val_df,   "val"),                 batch_size=config["batch_size"], shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(TensorCacheDataset(test_df,  "test"),                batch_size=config["batch_size"], shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device} ({torch.cuda.device_count()} GPUs)")

model = models.get_model(config["model"], weights="DEFAULT")
model.classifier = nn.Sequential(
    nn.Dropout(p=config["dropout"]),
    nn.Linear(model.classifier[1].in_features, config["num_classes"])
)
model = model.to(device)

Using: cuda (2 GPUs)
Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth


100%|██████████| 208M/208M [00:01<00:00, 202MB/s]  


In [10]:
def run_epoch(model, loader, optimizer, criterion, scaler, train=True, accumulation_steps=1):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.set_grad_enabled(train):
        for i, (imgs, labels) in enumerate(loader):
            imgs, labels = imgs.to(device), labels.to(device)

            with torch.autocast(device_type="cuda"):
                logits = model(imgs)
                loss   = criterion(logits, labels) / accumulation_steps

            if train:
                scaler.scale(loss).backward()
                if (i + 1) % accumulation_steps == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

            total_loss += loss.item() * accumulation_steps * len(labels)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += len(labels)

    return total_loss / total, correct / total

def log_confusion_matrix(model, loader, epoch):
    all_preds, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for imgs, labels in loader:
            preds = model(imgs.to(device)).argmax(1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    wandb.log({
        "confusion_matrix": wandb.plot.confusion_matrix(
            y_true=all_labels,
            preds=all_preds,
            class_names=REGIONS
        ),
        "epoch": epoch
    })

In [ ]:
checkpoint_path = Path("/kaggle/working/best_model.pt")

try:
    wandb.init(project="geo-classifier-quebec", config=config)
    cfg = wandb.config
    wandb.watch(model, log="gradients", log_freq=100)

    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    scaler    = torch.cuda.amp.GradScaler()
    best_val_acc = 0.0

    # ── Phase 1: head only ──────────────────────────────────────────
    print("Phase 1 — head only")
    for p in model.features.parameters():
        p.requires_grad = False

    optimizer = torch.optim.AdamW(
        model.classifier.parameters(),
        lr=cfg.lr_head, weight_decay=cfg.weight_decay
    )

    for epoch in range(cfg.epochs_head):
        train_loss, train_acc = run_epoch(model, train_loader, optimizer, criterion, scaler, train=True,  accumulation_steps=cfg.accumulation_steps)
        val_loss,   val_acc   = run_epoch(model, val_loader,   None,      criterion, scaler, train=False, accumulation_steps=cfg.accumulation_steps)

        print(f"[Phase 1 | {epoch+1}/{cfg.epochs_head}] "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        wandb.log({
            "epoch":       epoch,
            "train/loss":  train_loss,
            "train/acc":   train_acc,
            "val/loss":    val_loss,
            "val/acc":     val_acc,
            "lr":          optimizer.param_groups[0]["lr"],
            "phase":       1,
        })

    # ── Phase 2: full fine-tune ─────────────────────────────────────
    print("\nPhase 2 — full fine-tune")
    for p in model.features.parameters():
        p.requires_grad = True

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr_finetune, weight_decay=cfg.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.epochs_finetune
    )

    for epoch in range(cfg.epochs_finetune):
        global_epoch = cfg.epochs_head + epoch

        train_loss, train_acc = run_epoch(model, train_loader, optimizer, criterion, scaler, train=True)
        val_loss,   val_acc   = run_epoch(model, val_loader,   None,      criterion, scaler, train=False)
        scheduler.step()

        print(f"[Phase 2 | {epoch+1}/{cfg.epochs_finetune}] "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        wandb.log({
            "epoch":       global_epoch,
            "train/loss":  train_loss,
            "train/acc":   train_acc,
            "val/loss":    val_loss,
            "val/acc":     val_acc,
            "lr":          scheduler.get_last_lr()[0],
            "phase":       2,
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), checkpoint_path)
        
            artifact = wandb.Artifact("best-model", type="model")
            artifact.add_file(str(checkpoint_path))
            wandb.log_artifact(artifact)
            print(f"  ✓ best model saved (val_acc={val_acc:.4f})")
            
        if (epoch + 1) % 5 == 0:
            log_confusion_matrix(model, val_loader, global_epoch)

    # ── Log best model as artifact ──────────────────────────────────
    artifact = wandb.Artifact("best-model", type="model")
    artifact.add_file(str(checkpoint_path))
    wandb.log_artifact(artifact)
    print(f"\nDone. Best val_acc: {best_val_acc:.4f}")

finally:
    wandb.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: simon-bouchard31 (simon-bouchard31-self-employed) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Phase 1 — head only


/tmp/ipykernel_57/2643627303.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()


[Phase 1 | 1/5] train_loss=2.2004 train_acc=0.3958 val_loss=1.8407 val_acc=0.5228
[Phase 1 | 2/5] train_loss=1.8808 train_acc=0.5027 val_loss=1.6944 val_acc=0.5685
[Phase 1 | 3/5] train_loss=1.8111 train_acc=0.5335 val_loss=1.6631 val_acc=0.5685
[Phase 1 | 4/5] train_loss=1.7632 train_acc=0.5467 val_loss=1.6106 val_acc=0.6066
[Phase 1 | 5/5] train_loss=1.7490 train_acc=0.5455 val_loss=1.5931 val_acc=0.6100

Phase 2 — full fine-tune
[Phase 2 | 1/20] train_loss=1.3527 train_acc=0.7026 val_loss=1.0883 val_acc=0.8075
  ✓ best model saved (val_acc=0.8075)
[Phase 2 | 2/20] train_loss=0.9975 train_acc=0.8473 val_loss=0.9467 val_acc=0.8672
  ✓ best model saved (val_acc=0.8672)
[Phase 2 | 3/20] train_loss=0.8421 train_acc=0.9154 val_loss=0.9315 val_acc=0.8705
  ✓ best model saved (val_acc=0.8705)
[Phase 2 | 4/20] train_loss=0.7492 train_acc=0.9525 val_loss=0.8996 val_acc=0.8813
  ✓ best model saved (val_acc=0.8813)
[Phase 2 | 5/20] train_loss=0.6947 train_acc=0.9753 val_loss=0.8866 val_acc=0.88

In [30]:
wandb.init(project="geo-classifier-quebec", job_type="eval")

artifact = wandb.use_artifact("simon-bouchard31-self-employed/geo-classifier-quebec/best-model:v9", type="model")
artifact_dir = artifact.download()
model.load_state_dict(torch.load(f"{artifact_dir}/best_model.pt"))
model = model.float()

wandb: Downloading large artifact 'best-model:v9', 203.23MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.2 (911.7MB/s)


In [31]:
model.eval()
criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])
scaler    = torch.cuda.amp.GradScaler()

test_loss, test_acc = run_epoch(model, test_loader, None, criterion, scaler, train=False)
log_confusion_matrix(model, test_loader, epoch="test")
wandb.log({"test/loss": test_loss, "test/acc": test_acc})
print(f"Test accuracy: {test_acc:.4f}")

/tmp/ipykernel_57/3118384494.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()


RuntimeError: DataLoader worker (pid(s) 172, 173, 174, 175) exited unexpectedly

In [22]:
!pip install onnxruntime onnxscript onnxconverter-common -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 84.2 MB/s eta 0:00:00:00:0100:01


In [ ]:
import onnx
from onnxconverter_common import float16

model = model.cpu()

# Export to ONNX (model is already loaded and eval)
dummy = torch.randn(1, 3, 480, 480)
torch.onnx.export(
    model,
    dummy,
    "/kaggle/working/model.onnx",
    opset_version=17,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
)
print("Exported to ONNX")

# Sanity check
import onnxruntime as ort
import numpy as np

sess    = ort.InferenceSession("/kaggle/working/model.onnx")
ort_out = sess.run(["output"], {"input": dummy.numpy()})[0]
with torch.no_grad():
    pt_out = model(dummy).numpy()
print(f"Max diff: {np.abs(ort_out - pt_out).max():.6f}")  # should be < 1e-4

# Convert to FP16
model_fp16 = float16.convert_float_to_float16(onnx.load("/kaggle/working/model.onnx"))
onnx.save(model_fp16, "/kaggle/working/model_fp16.onnx")
print("Converted to FP16")

# Log to W&B
onnx_artifact = wandb.Artifact("quebec-classifier-onnx", type="model")
onnx_artifact.add_file("/kaggle/working/model_fp16.onnx")
wandb.log_artifact(onnx_artifact)
wandb.finish()